In [1]:
import pandas as pd
import re
import math
import json
import requests
from bs4 import BeautifulSoup
from io import StringIO
from variables import TOUR_NAME, POKEDATA_CSV, DAY1_ROUNDS, CATEGORY, LIMILESS_LABS_BASE_ENDPOINT, LIMITLESS_LABS_TOUR_ID

In [2]:
response = requests.get(LIMILESS_LABS_BASE_ENDPOINT.format(LIMITLESS_LABS_TOUR_ID, 'MA'))

In [3]:
deck_df = pd.DataFrame(response.json()['message'])

In [4]:
deck_df = pd.DataFrame({
    'Placement': deck_df['placement'],
    'Player': deck_df['name'],
    'Country': deck_df['country'],
    'Day 2': deck_df['day2'],
    'Deck': deck_df['deck_name'],
    'Variant': 'Standard'  # Set a constant value
})
deck_df = deck_df.dropna(subset=['Placement'])
deck_df['Placement'] = deck_df['Placement'].apply(lambda x: "Top {}".format(pow(2, math.ceil(math.log(x, 2)))))

In [5]:
def clean_name(input_string):
    result = re.sub(r'\s*\[.*?\]\s*', '', input_string)
    result = re.sub(r'STATIC SEATING \(\d+\)\s*', '', result)
    result = re.sub(r'>.*?>', '', result)
    result = re.sub(r'>TABLE \d+ ', '', result)
    return result

In [6]:
pairings_df = pd.read_csv(StringIO(requests.get(POKEDATA_CSV).content.decode('utf-8')), sep='\t', header=None, encoding='utf-8')
pairings_df.rename(columns={0:'Player',1:'Opponent',2:'Result',3:'Points',4:'Round'}, inplace=True)
pairings_df['Player'] = pairings_df['Player'].apply(clean_name)
pairings_df['Opponent'] = pairings_df['Opponent'].apply(clean_name)
pairings_df = pairings_df[(pairings_df['Opponent'] != 'BYE') & (pairings_df['Opponent'] != 'LATE')]

In [7]:
# Check missing players
player_index = 0
for player in pairings_df['Player'].unique():
    if player not in deck_df['Player'].unique():
        print(player_index, player)
    player_index+=1

3789 Josh Stapler
3790 Luis Olivera
3791 wilsonn molina
3792 Roberto Martínez
3793 Brian Wade
3794 Pepijn Korst
3795 Edwin Torres
3796 Lindsey Bennett


In [8]:
# Step 1: Get list of players who appear more than once (case-insensitive)
players_lower = deck_df['Player'].str.lower()
players_more_than_once = players_lower.value_counts()
players_more_than_once = players_more_than_once[players_more_than_once > 1].index.tolist()

# Step 2: Process and rename
for i in deck_df.index:
    player = deck_df.at[i, 'Player']
    player_lower = player.lower()

    if player_lower in players_more_than_once:
        new_name = f"{player} {i + 1}"
        deck_df.at[i, 'Player'] = new_name

        first_round = True
        # Filter pairings_df by case-insensitive match
        matching_rows = pairings_df[pairings_df['Player'].str.lower() == player_lower]

        for index, row in matching_rows.iterrows():
            if row['Round'] == 1:
                if first_round:
                    first_round = False
                else:
                    break

            # Update the player's name in pairings
            pairings_df.loc[index, 'Player'] = new_name

            # Update their opponent’s row as well
            opponent_lower = row['Opponent'].lower()
            pairings_df.loc[
                (pairings_df['Player'].str.lower() == opponent_lower) & 
                (pairings_df['Round'] == row['Round']),
                'Opponent'
            ] = new_name


In [9]:
players_more_than_once

['jacob smith',
 'richard nguyen',
 'gabriel perez',
 'josh mills',
 'robert jones',
 'hector ibarra',
 'ronald sharpsteen',
 'kevin lin',
 'billy crisp',
 'steven martinez',
 'mario martinez',
 'eric mei',
 'albert lee',
 'daniel powell',
 'will johnson']

In [10]:
with pd.ExcelWriter(f'datasets/{TOUR_NAME}_{CATEGORY}.xlsx') as writer:
    pairings_df.to_excel(writer, sheet_name='pairings', index=False)
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    # matchups_df.to_excel(writer, sheet_name='matchups', index=False)